In [17]:
%pip  install groq langchain langchain-groq langchain-community gradio langchain-huggingface python-dotenv sentence-transformers faiss-cpu pypdf jq reportlab beautifulsoup4

   ---------------------------------------- 0.0/31.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/31.4 MB ? eta -:--:--
   ---------------------------------------- 0.3/31.4 MB ? eta -:--:--
    --------------------------------------- 0.5/31.4 MB 1.5 MB/s eta 0:00:22
   - -------------------------------------- 0.8/31.4 MB 1.5 MB/s eta 0:00:21
   - -------------------------------------- 0.8/31.4 MB 1.5 MB/s eta 0:00:21
   -- ------------------------------------- 1.6/31.4 MB 1.5 MB/s eta 0:00:20
   -- ------------------------------------- 1.8/31.4 MB 1.4 MB/s eta 0:00:22
   -- ------------------------------------- 2.1/31.4 MB 1.4 MB/s eta 0:00:22
   --- ------------------------------------ 2.4/31.4 MB 1.4 MB/s eta 0:00:21
   --- ------------------------------------ 2.6/31.4 MB 1.4 MB/s eta 0:00:21
   --- ------------------------------------ 2.9/31.4 MB 1.4 MB/s eta 0:00:21
   ---- ----------------------------------- 3.1/31.4 MB 1.4 MB/s eta 0:00:21
   ---- ------------


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from groq import Groq
from langchain_groq import ChatGroq

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError(
        "GROQ_API_KEY not found in environment variables."
    )

#load the HR Policy PDF

pdf_path ="the_nestle_hr_policy_pdf_2012.pdf"

if not os.path.exists(pdf_path):
    raise FileNotFoundError(
        f"PDF file not found at {pdf_path}"
    )

loader = PyPDFLoader(pdf_path)
documents = loader.load()

if not documents:
    raise ValueError(
        "No documents found in the PDF."
    )

print("PDF Pages Loaded",len(documents))

# llm = ChatGroq(
#     model="openai/gpt-oss-20b"
# )
# response =llm.invoke("hi")
# print(response.content)

PDF Pages Loaded 8


In [19]:
#chunk the documents into smaller pieces

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks=splitter.split_documents(documents)
print("chunks created",len(chunks))

chunks created 20


In [20]:
# creating embiddings for the chunks
from langchain_huggingface import HuggingFaceEmbeddings

embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True }                 

)

print ("embeddings created")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embeddings created


In [21]:
# working with fassi 

from langchain_community.vectorstores import FAISS
vector_store=FAISS.from_documents(chunks,embeddings)   


# retrieving it 

retriever = vector_store.as_retriever(
    search_kwargs={"k": 4}
)


# this line means that it will save the vector store locally in a folder named "faiss_hr_policy". You can later load this vector store for querying without needing to recreate it from scratch.
vector_store.save_local("faiss_hr_policy")
print("vector store created and saved locally")

vector store created and saved locally


In [22]:
#config with groq


llm =ChatGroq(
    model="openai/gpt-oss-20b"
)

In [23]:
# prompt template
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

prompt = ChatPromptTemplate.from_template("""
You are a helpful HR policy assistant for Nestlé.

Answer the user's question using ONLY the context below.
If the answer is not present in the context, say:
"I could not find this information in the provided HR policy."

Do not invent policies or facts.
Keep your answer clear and simple.

Context:
{context}

Question:
{question}

Answer:


""")



In [24]:
# building question-answering chain 

from langchain_core.output_parsers import StrOutputParser
qa_chain = prompt | llm | StrOutputParser()


def chatbot(message, history):
    if not message.strip():
        return "Please enter a question."

    # Find relevant PDF chunks
    relevant_docs = retriever.invoke(message)

    if not relevant_docs:
        return "I could not find relevant information in the HR policy."

    # Combine retrieved text
    context = "\n\n".join(
        doc.page_content for doc in relevant_docs
    )

    # Generate answer
    answer = qa_chain.invoke({
        "context": context,
        "question": message
    })

    return answer


In [ ]:
question = "What is the exact number of annual leave days allowed to employees?"
answer = chatbot(question,[])
print(answer)

I could not find this information in the provided HR policy.


In [30]:

import gradio as gr
demo = gr.ChatInterface(
    fn=chatbot,
    title="Nestlé HR Policy Chatbot",
    description=(
        "Ask questions about the HR policy PDF. "
        "Answers are generated using the retrieved document text."
    ),
    examples=[
        "What is the leave policy?",
        "What are the working hours?",
        "What does the HR policy say about holidays?"
    ]
)


In [31]:
if __name__ == "__main__":
    demo.launch()


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
